In [67]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [68]:
df = pd.read_csv("/content/covid_toy.csv")

In [69]:
df.sample(10)

,age,gender,fever,cough,city,has_covid
81,65,Male,99.0,Mild,Delhi,No
18,64,Female,98.0,Mild,Bangalore,Yes
24,13,Female,100.0,Strong,Kolkata,No
41,82,Male,NaN,Mild,Kolkata,Yes
68,54,Female,104.0,Strong,Kolkata,No
99,10,Female,98.0,Strong,Kolkata,Yes
11,65,Female,98.0,Mild,Mumbai,Yes
5,84,Female,NaN,Mild,Bangalore,Yes
93,27,Male,100.0,Mild,Kolkata,Yes
94,79,Male,NaN,Strong,Kolkata,Yes


In [70]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


# Without Column Tranform Used

In [71]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2, random_state=42)

### So I can see in fever column has 10 null values, so I can replace them by mean.

In [72]:
from sklearn.impute import SimpleImputer
si = SimpleImputer()

In [73]:
fever_si = si.fit(X_train[['fever']])

In [74]:
X_train_fever = fever_si.transform(X_train[['fever']])
X_test_fever = fever_si.transform(X_test[['fever']])

In [75]:
df['cough'].unique()

array(['Mild', 'Strong'], dtype=object)

In [76]:
df['city'].unique()

array(['Kolkata', 'Delhi', 'Mumbai', 'Bangalore'], dtype=object)

In [77]:
df['gender'].unique()

array(['Male', 'Female'], dtype=object)

### Now convert these catagorical data into numerical representation.

In [78]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [79]:
ohe = OneHotEncoder(drop='first', sparse_output=False)
ode = OrdinalEncoder(categories=[['Mild', 'Strong']])

In [80]:
ohe_gender_city = ohe.fit(X_train[['gender', 'city']])

In [81]:
X_train_gender_city = ohe_gender_city.transform(X_train[['gender', 'city']])
X_test_gender_city = ohe_gender_city.transform(X_test[['gender', 'city']])

In [82]:
X_train_cough = ode.fit_transform(X_train[['cough']])
X_test_cough = ode.fit_transform(X_test[['cough']])

### Now I have cancatinate all of them.
  - Gender
  - City
  - Cough
  - Fever

  - **Age** make to separate then concatinate


In [83]:
X_train_age = X_train.drop(columns=['gender', 'city', 'cough', 'fever']).values
X_test_age = X_test.drop(columns=['gender', 'city', 'cough', 'fever']).values

In [84]:
X_train_new = np.concatenate((X_train_age, X_train_cough, X_train_gender_city, X_train_fever), axis=1)
X_test_new = np.concatenate((X_test_age, X_test_cough, X_test_gender_city, X_test_fever), axis=1)

In [85]:
X_train.shape

(80, 5)

In [86]:
X_train_new.shape

(80, 7)

### Why this number of columns changed?
  - Gender = Male & Female => [convert into 2 columns]
  - City = Mumbai, Kolkata and so on => convert into 4 columns
  - Cough = Mild & strong => convert into 2 columns

  - Total new columns = 2 + 4 + 2 = 8 how many old columns remove 3 so, X_train.shape was 5 coulmns remove 3 it will be 2, and now add 8 so it should be 10.
  - But here we remove or avoid multicollinearity so we take -1 columns from each so for gender take 2-1 = 1 coulmn, for City = 4-1 = 3 columns and cough took 1 column. So total 5 coulmns. Thats why it shows 2 + 5 = 7 coulmns.
     


# Column Tranform Used

In [87]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
42,27,Male,100.0,Mild,Delhi,Yes
78,11,Male,100.0,Mild,Bangalore,Yes
66,51,Male,104.0,Mild,Kolkata,No
29,34,Female,NaN,Strong,Mumbai,Yes
21,73,Male,98.0,Mild,Bangalore,Yes


### As it is, our actual dataset. Now we apply columnTransformer.

In [88]:
from sklearn.compose import ColumnTransformer

In [89]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [90]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2, random_state=42)

In [91]:
X_train.head(2)

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata


### Once split into train and test then we just handel other part like missing values, catagorical values.

In [92]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [93]:
# Now Missing values Impute
trn_missing = ColumnTransformer(transformers=[
    ('si', SimpleImputer(), [2])
], remainder='passthrough')

In [94]:
# One Hot Encoding
trn_ohe = ColumnTransformer(transformers=[
    ('ohe_gender_city', OneHotEncoder(drop='first', sparse_output=False), [1, 4]),
], remainder='passthrough')

In [95]:
# Ordinal Encoder -> Cough = df['cough'].qnique()
trn_ordinal = ColumnTransformer(transformers=[
    ('ode_cough', OrdinalEncoder(categories=[['Mild', 'Strong']]), [3])
], remainder='passthrough')


### This is Extra Pipeline also inculde here, that will give wrong because pipeline is used for passed one out as an input of next one.

In [96]:
from sklearn.pipeline import Pipeline

In [99]:
pipe = Pipeline([
    ('trn_missing', trn_missing),
    ('trn_ordinal', trn_ordinal),
    ('trn_ohe', trn_ohe)
])

In [100]:
X_train_pipe = pipe.fit_transform(X_train)
X_test_pipe = pipe.transform(X_test)

In [101]:
X_train.shape

(80, 5)

In [102]:
X_train_pipe.shape

(80, 12)

In [107]:
import numpy as np
# To view a few random rows from the numpy array
random_indices = np.random.choice(X_train_pipe.shape[0], 5, replace=False)
X_train_pipe[random_indices]

array([[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 56, 'Female'],
       [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 42, 'Female'],
       [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 51, 'Male'],
       [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 46, 'Male'],
       [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 79, 'Male']],
      dtype=object)

### OR

In [ ]:
# Now in one line we have write all transformer in one line and which column you want to apply the transform.

transform = ColumnTransformer(transformers=[
    ('trn1', SimpleImputer(), ['fever']),
    ('trn2', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city']),
    ('trn3', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough'])
], remainder='passthrough')  # <== remainder Passthrough means rest of the columns pass as it is.

In [ ]:
X_train_new_trans = transform.fit_transform(X_train)
X_test_new_trans  = transform.transform(X_test)

In [ ]:
X_train.shape

In [ ]:
X_train_new_trans.shape